# 🏙️ Seismic-CAIN-GAN: Multi-City Urban Site Planning
## Elazığ + İstanbul için Sismik Farkındalıklı GAN

**Temel makale:** *"Automated site planning using CAIN-GAN model"* (Automation in Construction, Elsevier 2024)

**Bu çalışmanın yenilikleri:**
- ✅ Multi-city support (Elazığ + İstanbul)
- ✅ Sismik attention gate (AFAD PGA)
- ✅ Topografi farkındalığı (Copernicus DEM)
- ✅ City-conditional normalization
- ✅ Cross-city evaluation

---

## ⚙️ Çalıştırma
1. **Runtime → Change runtime type → GPU (T4/A100)**
2. **Runtime → Run all**
3. Synthetic veri ile baştan sona test edebilirsiniz

## 1️⃣ Ortam Kurulumu

In [ ]:
import sys, os
print(f'Python: {sys.version}')
print(f'CWD:    {os.getcwd()}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('\n✅ Google Drive bağlandı')
except ImportError:
    print('\n⚠️ Colab dışında çalışıyor')

## 2️⃣ Repo Klonlama

In [ ]:
GITHUB_USERNAME = 'ilker-23'
REPO_NAME = 'cain-gan-urban-design'

!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
%cd {REPO_NAME}
!ls -la

## 3️⃣ Bağımlılıklar

In [ ]:
!pip install -q -r requirements.txt
print('✅ Bağımlılıklar yüklendi')

In [ ]:
import torch, torchvision, albumentations, numpy as np, PIL
print(f'PyTorch:        {torch.__version__}')
print(f'Torchvision:    {torchvision.__version__}')
print(f'Albumentations: {albumentations.__version__}')
print(f'NumPy:          {np.__version__}')
print(f'Pillow:         {PIL.__version__}')

## 4️⃣ GPU Kontrolü

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'✅ GPU:         {torch.cuda.get_device_name(0)}')
    print(f'   Bellek:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'   CUDA:        {torch.version.cuda}')
    !nvidia-smi | head -20
else:
    print('❌ GPU yok! Runtime → Change runtime type → GPU')

## 5️⃣ Veri Hazırlığı (Synthetic Test Verisi)

Gerçek veri için **TURKISH_CITIES_GUIDE.md** dosyasını okuyun.
Bu hücre, pipeline'ı test etmek için Elazığ + İstanbul sahte verisi üretir.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('./data')
cities = ['elazig', 'istanbul']
splits = ['train', 'val', 'test']
subdirs = ['site_context', 'planning_guidance', 'neighboring_footprints',
           'mask', 'seismic', 'dem', 'footprint_target', 'height_target']

for city in cities:
    for split in splits:
        for sub in subdirs:
            (DATA_ROOT / city / split / sub).mkdir(parents=True, exist_ok=True)

print('📁 Dizin yapısı:')
!find data -maxdepth 4 -type d | sort | head -30

In [ ]:
import numpy as np
from PIL import Image

def generate_city_sample(city: str, sample_id: str, split: str):
    '''Şehir-spesifik sahte CAIN-GAN örneği üret.'''
    np.random.seed(hash(f'{city}_{sample_id}') % 2**32)
    
    base = Path(f'data/{city}/{split}')
    
    # Şehir karakteristikleri
    if city == 'elazig':
        density = 0.35   # düşük yoğunluk
        seismic_base = 0.7  # yüksek deprem riski (1. derece)
        elevation_var = 0.6  # plato şehir
    else:  # istanbul
        density = 0.65   # yüksek yoğunluk
        seismic_base = 0.5  # orta-yüksek (KAF)
        elevation_var = 0.8  # tepelik
    
    # 1. Site context
    site = np.random.choice([0, 64, 128, 192], size=(256, 256),
                              p=[0.6, 0.25, 0.10, 0.05]).astype(np.uint8)
    Image.fromarray(site).save(base / 'site_context' / f'{sample_id}.png')
    
    # 2. Planning guidance (RGB one-hot)
    planning = np.random.randint(0, 4, size=(256, 256))
    planning_rgb = np.zeros((256, 256, 3), dtype=np.uint8)
    planning_rgb[:, :, 0] = (planning == 0) * 255  # konut
    planning_rgb[:, :, 1] = (planning == 1) * 255  # ticaret
    planning_rgb[:, :, 2] = (planning == 2) * 255  # sanayi
    Image.fromarray(planning_rgb).save(base / 'planning_guidance' / f'{sample_id}.png')
    
    # 3. Neighboring footprints
    footprints = (np.random.rand(256, 256) > (1 - density)).astype(np.uint8) * 255
    Image.fromarray(footprints).save(base / 'neighboring_footprints' / f'{sample_id}.png')
    
    # 4. Mask (merkez 80x80 tasarım alanı)
    mask = np.ones((256, 256), dtype=np.uint8) * 255
    mask[88:168, 88:168] = 0
    Image.fromarray(mask).save(base / 'mask' / f'{sample_id}.png')
    
    # 5. Sismik risk (radyal gradient ~ fay hattına mesafe)
    yy, xx = np.mgrid[0:256, 0:256]
    center_x, center_y = np.random.randint(50, 200, size=2)
    dist = np.sqrt((xx - center_x)**2 + (yy - center_y)**2) / 200
    seismic = np.clip((seismic_base + 0.3 * (1 - dist) + np.random.randn(256, 256) * 0.05) * 255,
                       0, 255).astype(np.uint8)
    Image.fromarray(seismic).save(base / 'seismic' / f'{sample_id}.png')
    
    # 6. DEM
    elevation = np.clip(elevation_var * (np.sin(xx/30) * np.cos(yy/40) + 1) / 2 * 255,
                         0, 255).astype(np.uint8)
    Image.fromarray(elevation).save(base / 'dem' / f'{sample_id}.png')
    
    # 7. Footprint target
    fp_target = (np.random.rand(256, 256) > (1 - density * 0.8)).astype(np.uint8) * 255
    Image.fromarray(fp_target).save(base / 'footprint_target' / f'{sample_id}.png')
    
    # 8. Height target
    height = (footprints / 255 * np.random.randint(50, 200, size=(256, 256))).astype(np.uint8)
    Image.fromarray(height).save(base / 'height_target' / f'{sample_id}.png')


# Her şehir için örnekler
counts = {'train': 20, 'val': 5, 'test': 5}
for city in cities:
    for split, n in counts.items():
        for i in range(n):
            generate_city_sample(city, f'sample_{i:04d}', split)
        print(f'✅ {city}/{split}: {n} örnek')

print('\n🎉 Multi-city synthetic veri hazır!')

## 6️⃣ Multi-City Dataset Test

In [ ]:
from multi_city_dataset import TurkishCityDataset, create_multi_city_dataloaders, CITY_INDEX

# Multi-city loaders
loaders = create_multi_city_dataloaders(
    data_root='./data',
    cities=['elazig', 'istanbul'],
    batch_size=4,
    num_workers=2,
    stage='footprint',
    use_seismic=True,
    use_topography=True,
    use_city_embedding=True,
)

print('📊 Loaders:')
for key, loader in loaders.items():
    print(f'  {key}: {len(loader.dataset)} örnek, {len(loader)} batch')

# Bir batch al
batch = next(iter(loaders['train']))
print(f"\n📦 Batch:")
print(f"  conditional_inputs: {batch['conditional_inputs'].shape}")
print(f"  footprint:          {batch['footprint'].shape}")
print(f"  cities:             {batch['city']}")
print(f"  city_index:         {batch['city_index'].tolist()}")

In [ ]:
import torch
from seismic_extension import SeismicCAINGenerator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Conditional kanal sayısı: site(1) + planning(3) + footprints(1) + mask(1) + seismic(1) + dem(1) + city(2) = 10
n_channels = batch['conditional_inputs'].shape[1]
print(f'Conditional kanal sayısı: {n_channels}')

gen = SeismicCAINGenerator(
    conditional_channels=n_channels,
    use_city_norm=True,
    num_cities=2,
).to(device)

n_params = sum(p.numel() for p in gen.parameters() if p.requires_grad)
print(f'Toplam parametre: {n_params:,}')

# Forward pass
conditional = batch['conditional_inputs'].to(device)
city_idx = batch['city_index'].to(device)
seismic_map = conditional[:, 6:7]  # 6. kanal sismik

with torch.no_grad():
    output = gen(conditional, seismic_map=seismic_map, city_index=city_idx)

print(f'✅ Generator çıktısı: {output.shape}')

## 7️⃣ Seismic-CAIN-GAN Eğitimi

⏰ Synthetic veri ile ~10-20 dakika.
Gerçek veri için epoch sayısını 50+'a çıkarın.

In [ ]:
from multi_city_training import MultiCityCAINTrainer

trainer = MultiCityCAINTrainer(
    data_root='./data',
    checkpoint_dir='./checkpoints',
    cities=['elazig', 'istanbul'],
    device='cuda' if torch.cuda.is_available() else 'cpu',
    batch_size=4,         # Synthetic için küçük
    num_workers=2,
    learning_rate=0.0002,
    lambda_seismic=10.0,
    lambda_terrain=5.0,
    use_seismic=True,
    use_topography=True,
    use_city_embedding=True,
)

trainer.train(
    num_epochs_footprint=5,   # Test için az (gerçek için 50+)
    num_epochs_height=5,
    save_interval=2,
)

## 8️⃣ Sonuçları Görselleştir

In [ ]:
import matplotlib.pyplot as plt
from multi_city_dataset import TurkishCityDataset

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for row, city in enumerate(['elazig', 'istanbul']):
    ds = TurkishCityDataset(data_root='./data', city=city, split='val',
                              stage='footprint', augment=False)
    sample = ds[0]
    cond = sample['conditional_inputs']
    target = sample['footprint']
    
    # 5 kanal göster: site, planning(R), footprints, seismic, DEM
    titles = ['Site Context', 'Planning (R)', 'Footprints', 'Seismic', 'DEM']
    channels = [0, 1, 4, 6, 7]  # indeksleri
    
    for col, (title, ch_idx) in enumerate(zip(titles, channels)):
        axes[row, col].imshow(cond[ch_idx].numpy(), cmap='gray' if col != 3 else 'hot')
        axes[row, col].set_title(f'{city.upper()} — {title}')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('multi_city_channels.png', dpi=100, bbox_inches='tight')
plt.show()
print('💾 Kaydedildi: multi_city_channels.png')

In [ ]:
# Eğitilmiş modelin tahminleri
import torch
import matplotlib.pyplot as plt
from multi_city_dataset import TurkishCityDataset
from seismic_extension import SeismicCAINGenerator

# Son checkpoint'i bul
from pathlib import Path
ckpts = sorted(Path('./checkpoints').glob('footprint_epoch_*.pth'))
if not ckpts:
    print('⚠️ Checkpoint bulunamadı, önce eğitimi çalıştırın')
else:
    ckpt = torch.load(ckpts[-1])
    n_ch = ckpt['config']['conditional_channels']
    
    gen = SeismicCAINGenerator(conditional_channels=n_ch, num_cities=2).to(device)
    gen.load_state_dict(ckpt['generator'])
    gen.eval()
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    
    for row, city in enumerate(['elazig', 'istanbul']):
        ds = TurkishCityDataset(data_root='./data', city=city, split='val',
                                  stage='footprint', augment=False)
        sample = ds[0]
        cond = sample['conditional_inputs'].unsqueeze(0).to(device)
        target = sample['footprint']
        city_idx = torch.tensor([sample['city_index']]).to(device)
        seismic = cond[:, 6:7]
        
        with torch.no_grad():
            pred = gen(cond, seismic_map=seismic, city_index=city_idx)
        
        axes[row, 0].imshow(cond[0, 0].cpu(), cmap='gray')
        axes[row, 0].set_title(f'{city.upper()} — Input')
        axes[row, 0].axis('off')
        
        axes[row, 1].imshow(target[0].cpu(), cmap='gray')
        axes[row, 1].set_title('Ground Truth')
        axes[row, 1].axis('off')
        
        axes[row, 2].imshow(pred[0, 0].cpu(), cmap='gray')
        axes[row, 2].set_title('Predicted')
        axes[row, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('multi_city_predictions.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('💾 Kaydedildi: multi_city_predictions.png')

## 9️⃣ Sonuçları Drive'a Kaydet

In [ ]:
import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/seismic_cain_gan')

if Path('/content/drive/MyDrive').exists():
    DRIVE.mkdir(exist_ok=True)
    
    if Path('./checkpoints').exists():
        shutil.copytree('./checkpoints', DRIVE / 'checkpoints', dirs_exist_ok=True)
        print(f'✅ Checkpoint\'ler: {DRIVE}/checkpoints')
    
    for img in Path('.').glob('*.png'):
        shutil.copy(img, DRIVE / img.name)
        print(f'✅ Görsel: {img.name}')
else:
    print('⚠️ Drive bağlı değil')

## 🎓 Sonraki Adımlar

### Gerçek Veri İçin
1. **[TURKISH_CITIES_GUIDE.md](./TURKISH_CITIES_GUIDE.md)** dosyasını okuyun
2. Elazığ + İstanbul verilerini toplayın (OSM, İBB, AFAD, Copernicus)
3. `data/{elazig,istanbul}/{train,val,test}/{8 alt dizin}` yapısına yerleştirin
4. Epoch sayısını **50+'a çıkarın**
5. `batch_size`'ı GPU'nuza göre ayarlayın (T4: 8, A100: 16-32)

### Cross-City Transfer Deneyi
```python
# 1. Elazığ'da eğit
trainer_e = MultiCityCAINTrainer(..., cities=['elazig'])
trainer_e.train(50, 50)

# 2. İstanbul'da fine-tune et
# (load_state_dict ile elazig modelinden başlat)
trainer_i = MultiCityCAINTrainer(..., cities=['istanbul'])
# ...
```

### SCI Makale İçin Önerilen Deneyler
1. **Ablation studies:** sismik/topografi/city-embedding'i sırayla kapatıp test et
2. **Cross-city evaluation:** Elazığ-train + İstanbul-test, ters yön
3. **Ölçek invariance:** model 256×256 + 512×512 patch'lerde test
4. **Hyperparameter sensitivity:** λ_seismic, λ_terrain için grid search

---

## 📚 Atıf

```bibtex
@article{jiang2024automated,
  title={Automated site planning using CAIN-GAN model},
  author={Jiang, F. and Ma, J. and Webster, C. and Wang, W. and Cheng, J.C.P.},
  journal={Automation in Construction},
  volume={159}, pages={105286}, year={2024},
  doi={10.1016/j.autcon.2024.105286}
}
```

🎓 Tez ve makale çalışmanızda başarılar!